In [ ]:
# Import torch
import torch

print(torch.__version__)

# TODO: Setup device agnostic code
device ="cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from torchvision import datasets
from torch import nn
import torchvision
from torchvision.transforms import ToTensor
train_data = torchvision.datasets.MNIST(
    root="data",
    train=True,
    transform=ToTensor(),
    download = True
)
test_data = torchvision.datasets.MNIST(
    root="data",
    train=False,
    transform=ToTensor(),
    download=True
)

In [ ]:
import matplotlib.pyplot as plt
import random
plt.figure(figsize=(10,10))
for i in range(6):
  random_number = random.randint(1,len(train_data))
  image , label = train_data[random_number]
  plt.subplot(3,3,i+1)
  plt.imshow(image.squeeze(),)
  plt.axis(False)



In [ ]:
from torch.utils.data import DataLoader
test_dataLoader = DataLoader(dataset=test_data,
                             batch_size=32,
                             shuffle=True,
                             )
train_dataLoader = DataLoader(dataset=train_data,
                              batch_size=32,
                              shuffle=True)
print(f"train: {train_dataLoader.batch_size} , test: {test_dataLoader.batch_size}")

In [ ]:
image.shape

In [ ]:
class_name = train_data.classes
class_name

In [ ]:
from torch.nn.modules.activation import ReLU
class MNESTV0(nn.Module):
  def __init__(self, input_shape,output_shape,hidden_unit):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  padding=1,
                  stride=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_unit,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_unit,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_unit,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )
    self.classifier= nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_unit*7*7,
                  out_features=output_shape)
    )
  def forward(self,x):
    x = self.conv_block_1(x)
    # print(x.shape)
    x = self.conv_block_2(x)
    # print(x.shape)
    x = self.classifier(x)
    return x

model_gpu = MNESTV0(input_shape=1,
                  output_shape=len(class_name),
                  hidden_unit=10
                  ).to(device)

model_cpu = MNESTV0(input_shape=1,
                  output_shape=len(class_name),
                  hidden_unit=10
                  ).cpu()

In [ ]:
x, y = train_data[0]
model_gpu(x.unsqueeze(0).to(device))

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer_gpu = torch.optim.SGD(model_gpu.parameters(),
                            lr=0.1)
optimizer_cpu = torch.optim.SGD(model_cpu.parameters(),
                                lr=0.1)

In [ ]:
!pip install torchmetrics
import torchmetrics
acc_fn_cpu = torchmetrics.Accuracy(task="multiclass" ,
                               num_classes=len(class_name)).to("cpu")
acc_fn_gpu = torchmetrics.Accuracy(task="multiclass" ,
                               num_classes=len(class_name)).to(device)

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device):
    train_loss, train_acc = 0, 0
    model.to(device)
    for batch, (X, y) in enumerate(data_loader):
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss
        train_acc += accuracy_fn(
                                 y_pred.argmax(dim=1),y) # Go from logits -> pred labels

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)
    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")

def test_step(data_loader: torch.utils.data.DataLoader,
              model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device):
    test_loss, test_acc = 0, 0
    model.to(device)
    model.eval() # put model in eval mode
    # Turn on inference context manager
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to GPU
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred = model(X)

            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_pred, y)
            test_acc += accuracy_fn(
                test_pred.argmax(dim=1),y
            )

        test_loss /= len(data_loader)
        test_acc /= len(data_loader)
        print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%\n")

In [ ]:
from timeit import default_timer as timer

def print_train_time(start: float, end: float, device: torch.device = None):
  total_time = end - start
  print(f"Train time on {device}: {total_time:.3f} seconds")

In [ ]:
from torch.nn.modules import loss
from tqdm.auto import tqdm
torch.manual_seed(42)
epochs = 5
start_timer_gpu=timer()
print("\n GPU \n")
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch} \n----------")
  train_step(model=model_gpu,
             data_loader=train_dataLoader,
             loss_fn=loss_fn,
             optimizer=optimizer_gpu,
             accuracy_fn=acc_fn_gpu,
             device=device)
  test_step(data_loader=test_dataLoader,
            model=model_gpu,
            loss_fn=loss_fn,
            accuracy_fn=acc_fn_gpu,
            device=device)
stop_timer_gpu=timer()
print_train_time(start=start_timer_gpu,end=stop_timer_gpu,device=device)


In [ ]:
torch.manual_seed(42)
epochs = 5
start_timer_cpu=timer()
print("\n CPU \n")
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch} \n----------")
  train_step(model=model_cpu,
             data_loader=train_dataLoader,
             loss_fn=loss_fn,
             optimizer=optimizer_cpu,
             accuracy_fn=acc_fn_cpu,
             device = "cpu")
  test_step(data_loader=test_dataLoader,
            model=model_cpu,
            loss_fn=loss_fn,
            accuracy_fn=acc_fn_cpu,
            device = "cpu")
stop_timer_cpu=timer()
print_train_time(start=start_timer_gpu,end=stop_timer_gpu,device="cpu")

In [ ]:
def make_Prediction (model:torch.nn.Module,
                     data : list,
                     device : torch.device = device):
  pred_probs=[]
  model.eval()
  with torch.inference_mode():
    for sample in data:
      sample = torch.unsqueeze(sample,dim=0).to(device)
      y_logit = model(sample)
      pred_prob = torch.softmax(y_logit.squeeze(),dim=0)

      pred_probs.append(pred_prob.cpu())
  return torch.stack(pred_probs)

In [ ]:
test_samples = []
test_labels = []

for sample, label in test_data:
    test_samples.append(sample)
    test_labels.append(label)
print(f"Test sample image shape: {test_samples[0].shape}\nTest sample label: {test_labels[0]} ({class_name[test_labels[0]]})")

In [ ]:
y_pred = make_Prediction(model=model_gpu,data=test_samples)
y_pred[:5]

In [ ]:
prediction= y_pred.argmax(dim=1)
prediction[1]
class_name[1]
test_labels[i]

In [ ]:
plt.figure(figsize=(9,9))
row = 3
col = 3
for i in range(9):
  random_number = random.randint(0,len(prediction))
  plt.subplot(row,col,i+1)
  plt.imshow(test_samples[random_number].squeeze(),cmap="gray")
  pred_label = class_name[prediction[i]]
  true_label = class_name[test_labels[i]]
  title_label = f"pred: {pred_label} | True: {true_label}"
  if pred_label ==true_label:
    plt.title(title_label,fontsize=10,c="g")
  else:
    plt.title(title_label,fontsize=10,c="r")
  plt.axis(False)

In [ ]:
!pip install mlxtend

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix
conf_mat = ConfusionMatrix(task="multiclass",
                              num_classes=len(class_name))
confmat_tensor= conf_mat(preds=prediction,target=test_data.targets)
fix,ax = plot_confusion_matrix(conf_mat=confmat_tensor.numpy(),
                               class_names=class_name,
                               colorbar=True,
                               figsize=(10,6))

In [ ]:
confmat_tensor

In [ ]:
images = torch.randn(1,3,64,64)
test_images = images[0]
print(f"Image batch shape: {images.shape} -> [batch_size, color_channels, height, width]")
print(f"Single image shape: {test_images.shape} -> [color_channels, height, width]")

In [ ]:
conv_layer = nn.Conv2d(in_channels=3,out_channels=10,kernel_size=9,stride=1,padding=4)
test = conv_layer(test_images)
test.shape

In [ ]:
from pathlib import Path
MODEL_PATH = Path("model")
MODEL_PATH.mkdir(parents=True,
                 exist_ok=True)
MODEL_NAME = "04_pytorch_computer_vision.pth"
MODEL_SAVE_PATH = MODEL_PATH/MODEL_NAME
print(f"save model to: {MODEL_SAVE_PATH}")
torch.save(obj = model_gpu.state_dict(),
           f=MODEL_SAVE_PATH)